# Oil Price Dataset Preprocessing

This notebook loads crude-oil price data, cleans it, and creates a model-ready CSV for the dashboard and predictor.


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_PATH = RAW_DIR / "oil_prices.csv"
PROCESSED_PATH = PROCESSED_DIR / "oil_prices_preprocessed.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH


If `data/raw/oil_prices.csv` already exists, the notebook uses it. Otherwise it tries to download Brent crude futures (`BZ=F`) from Yahoo Finance with `yfinance`.


In [ ]:
def load_or_download_oil_data(raw_path: Path) -> pd.DataFrame:
    if raw_path.exists() and raw_path.stat().st_size > 0:
        return pd.read_csv(raw_path)

    try:
        import yfinance as yf
    except ImportError as exc:
        raise ImportError(
            "Install yfinance or place an oil CSV at data/raw/oil_prices.csv."
        ) from exc

    oil_df = yf.download(
        "BZ=F",
        start="2000-01-01",
        progress=False,
        auto_adjust=False,
        multi_level_index=False,
    )
    if oil_df.empty:
        raise ValueError("No oil data was downloaded from Yahoo Finance.")

    oil_df = oil_df.reset_index()
    oil_df.to_csv(raw_path, index=False)
    return oil_df


raw_df = load_or_download_oil_data(RAW_PATH)
raw_df.head()


In [ ]:
def clean_oil_data(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    df.columns = [str(col).strip().lower().replace(" ", "_") for col in df.columns]

    if "date" not in df.columns:
        raise KeyError("Oil data must contain a Date column.")

    price_candidates = ["adj_close", "close", "price", "value"]
    price_col = next((col for col in price_candidates if col in df.columns), None)
    if price_col is None:
        raise KeyError("Oil data must contain one of: Adj Close, Close, Price, or Value.")

    cleaned = df[["date", price_col]].rename(columns={price_col: "oil_price_usd"})
    cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
    cleaned["oil_price_usd"] = pd.to_numeric(cleaned["oil_price_usd"], errors="coerce")
    cleaned = cleaned.dropna(subset=["date", "oil_price_usd"])
    cleaned = cleaned.drop_duplicates(subset="date").sort_values("date")

    cleaned["oil_return_pct"] = cleaned["oil_price_usd"].pct_change() * 100
    cleaned["oil_ma_7"] = cleaned["oil_price_usd"].rolling(window=7, min_periods=1).mean()
    cleaned["oil_ma_30"] = cleaned["oil_price_usd"].rolling(window=30, min_periods=1).mean()
    cleaned["oil_volatility_30"] = cleaned["oil_return_pct"].rolling(window=30, min_periods=2).std()
    cleaned["oil_return_pct"] = cleaned["oil_return_pct"].fillna(0)

    return cleaned.reset_index(drop=True)


clean_df = clean_oil_data(raw_df)
clean_df.head()


In [ ]:
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved preprocessed oil data to: {PROCESSED_PATH}")
print(f"Rows: {len(clean_df):,}")
clean_df.tail()
